## Exemple de représentation de tarifs

In [1]:
import sys
from pathlib import Path

# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))

In [2]:
import json
from source.tariff import Tariff, TariffElement, TariffElements, TariffDimensionType, TariffRestrictions, PriceComponent
from source.utils import Format, Price

## Exemple du modèle OCPI 2.3 avec différents formats
Trois formats sont interchangeables et peuvent être utilisés pour construire un objet Tariff:
- format JSON OCPI
- format JSON simplifié
- format texte réduit (moins de 256 caractères)

In [3]:
#exemple de création d'un tarif avec les entités OCPI
price_component1 = PriceComponent(type=TariffDimensionType.ENERGY, price=Price(amount=0.30))
price_component2 = PriceComponent(type=TariffDimensionType.FLAT, price=Price(amount=1.20))
price_component3 = PriceComponent(type=TariffDimensionType.ENERGY, price=Price(amount=0.10))
price_component4 = PriceComponent(type=TariffDimensionType.FLAT, price=Price(amount=1.50))
tariff_element1 = TariffElement(
        price_components=[price_component1, price_component2],
        restrictions=TariffRestrictions.from_json({"days_of_week": ["MONDAY", "TUESDAY"],"start_date": "2024-01-01"})
    )
tariff_element2 = TariffElement(price_components=[price_component3, price_component4])
tariff_elements = TariffElements([tariff_element1, tariff_element2])
tariff = Tariff(id="tariff1", elements=tariff_elements)


In [4]:
# Affichage du tarif au format JSON OCPI 2.3
print(json.dumps(tariff.to_json(), indent=2))

{
  "id": "tariff1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        },
        {
          "type": "FLAT",
          "price": 1.2
        }
      ],
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.1
        },
        {
          "type": "FLAT",
          "price": 1.5
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "tax_included": "YES"
}


In [5]:
# Affichage du tarif au format JSON simplifié
print(json.dumps(tariff.to_json(simple=True), indent=2))

{
  "id": "tariff1",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.3,
        "FLAT": 1.2
      },
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.1,
        "FLAT": 1.5
      }
    }
  ]
}


In [6]:
# Affichage du tarif dans un format texte compacté (uniquement la partie elements)
tariff_text = tariff.to_string()
tariff_text

'EN30+FL120+J=LuMa+D>2024-01-01|EN10+FL150'

In [7]:
# Affichage du tarif au format OCPI 2.3 à partir de la partie texte compacté uniquement
print(json.dumps(Tariff.convert(tariff_text, Format.JSON_OCPI), indent=2))

{
  "id": "undefined",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.3
        },
        {
          "type": "FLAT",
          "price": 1.2
        }
      ],
      "restrictions": {
        "days_of_week": [
          "MONDAY",
          "TUESDAY"
        ],
        "start_date": "2024-01-01"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.1
        },
        {
          "type": "FLAT",
          "price": 1.5
        }
      ]
    }
  ],
  "type": "AD_HOC_PAYMENT",
  "tax_included": "YES"
}


## Exemple d'un tarif variable suivant les tranches horaires

- 0,22 €/kWh de 00:00 à 04:00
- 0,22 €/kWh de 04:00 à 08:00
- 0,49 €/kWh de 08:00 à 20:00
- 0,35 €/kWh de 20:00 à 00:00
- Frais de congestion (parking sans recharge) 0,50 €/min

In [8]:
tariff_text = """
  EN22 + PT3000 + T>00:00 + T<04:00
  EN22 + PT3000 + T>04:00 + T<08:00
  EN49 + PT3000 + T>08:00 + T<20:00
  EN35 + PT3000"""

tariff = Tariff.from_string(tariff_text, id="t1", tariff_alt_text="exemple de tarif par plage horaire")


In [9]:
print(f"tarif texte compact (sur une ligne) : \n{tariff.to_string()} - {len(tariff.to_string())} caractères")

tarif texte compact (sur une ligne) : 
EN22+PT3000+T>00:00+T<04:00|EN22+PT3000+T>04:00+T<08:00|EN49+PT3000+T>08:00+T<20:00|EN35+PT3000 - 95 caractères


In [10]:
print(f"tarif json (compact) : \n{json.dumps(tariff.to_json(simple=True), indent=2)}")

tarif json (compact) : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": {
        "ENERGY": 0.22,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "00:00",
        "end_time": "04:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.22,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "04:00",
        "end_time": "08:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.49,
        "PARKING_TIME": 30.0
      },
      "restrictions": {
        "start_time": "08:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": {
        "ENERGY": 0.35,
        "PARKING_TIME": 30.0
      }
    }
  ],
  "tariff_alt_text": "exemple de tarif par plage horaire"
}


In [11]:
print(f"tarif json (OCPI 2.3) : \n{json.dumps(tariff.to_json(), indent=2)}")

tarif json (OCPI 2.3) : 
{
  "id": "t1",
  "elements": [
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.22
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "00:00",
        "end_time": "04:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.22
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "04:00",
        "end_time": "08:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENERGY",
          "price": 0.49
        },
        {
          "type": "PARKING_TIME",
          "price": 30.0
        }
      ],
      "restrictions": {
        "start_time": "08:00",
        "end_time": "20:00"
      }
    },
    {
      "price_components": [
        {
          "type": "ENER